In [2]:
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from hmmlearn.hmm import GaussianHMM
from arch import arch_model
from pykalman import KalmanFilter
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
import warnings

# Suppress convergence warnings for a cleaner dashboard
warnings.filterwarnings("ignore")

# --- 1. UNIFIED FEATURE LAYER (With MultiIndex Fix) ---
def fetch_and_preprocess(ticker, period="3y"):
    df = yf.download(ticker, period=period, interval="1d")
    
    if df.empty:
        return None

    # FIX: Flatten MultiIndex columns if they exist
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # FIX: Ensure 'Adj Close' exists (fallback to 'Close')
    if 'Adj Close' not in df.columns:
        df['Adj Close'] = df['Close']
        
    df['Return'] = df['Adj Close'].pct_change()
    
    # Technical Indicators
    df['MA20'] = df['Adj Close'].rolling(20).mean()
    delta = df['Adj Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    df['RSI'] = 100 - (100 / (1 + rs))
    
    return df.dropna()

# --- 2. MODEL-SPECIFIC OUTPUTS ---
def run_quant_suite(df):
    returns = df['Return'].values.reshape(-1, 1)
    prices = df['Adj Close'].values
    
    # A. Linear Regression (r_hat_LR)
    X_lr = np.arange(len(df)).reshape(-1, 1)
    lr_model = LinearRegression().fit(X_lr[-60:], returns[-60:])
    r_hat_lr = lr_model.predict([[len(df)]])[0][0]

    # B. ARIMA (r_hat_ARIMA)
    try:
        arima_model = ARIMA(returns, order=(1, 1, 1)).fit()
        r_hat_arima = arima_model.forecast(steps=1)[0]
    except:
        r_hat_arima = r_hat_lr # Fallback

    # C. GARCH (Sigma_t) - Scale by 100 for stability
    garch_model = arch_model(returns * 100, vol='Garch', p=1, q=1, dist='Normal')
    garch_res = garch_model.fit(disp='off')
    sigma_t = np.sqrt(garch_res.forecast(horizon=1).variance.values[-1, 0]) / 100

    # D. Hidden Markov Model (P_St)
    hmm = GaussianHMM(n_components=2, covariance_type="full", n_iter=100)
    hmm.fit(returns)
    probs = hmm.predict_proba(returns)
    bull_idx = np.argmax(hmm.means_) 
    prob_bull = probs[-1, bull_idx]

    # E. Kalman Filter (x_hat_t)
    kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1], 
                      initial_state_mean=prices[0], initial_state_covariance=1, 
                      observation_covariance=1, transition_covariance=0.01)
    state_means, _ = kf.filter(prices)
    smoothed_trend = state_means[-1][0]

    return r_hat_lr, r_hat_arima, sigma_t, prob_bull, smoothed_trend

# --- 3. SCENARIO LAYER (MONTE CARLO) ---
def run_monte_carlo(current_price, sigma, days=5, sims=500):
    returns_sim = np.random.normal(0, sigma, (sims, days))
    price_paths = current_price * np.exp(np.cumsum(returns_sim, axis=1))
    var_5 = np.percentile(price_paths[:, -1], 5)
    conf_95 = np.percentile(price_paths[:, -1], 95)
    return var_5, conf_95, price_paths

# --- 4. DASHBOARD UI ---
st.set_page_config(page_title="Unified Alpha Quant Engine", layout="wide")
st.title("⚡ Unified Alpha Quant Engine")

ticker = st.sidebar.text_input("Ticker Symbol", value="AAPL")
risk_k = st.sidebar.slider("Risk Scaling (k)", 0.01, 1.0, 0.2)

df = fetch_and_preprocess(ticker)

if df is not None:
    r_lr, r_arima, sigma_t, prob_bull, k_trend = run_quant_suite(df)

    # Combining Signal per your formula
    # Signal_t = w1*r_lr + w2*r_arima + (Placeholder for NN)
    signal_base = (0.5 * r_lr) + (0.5 * r_arima)
    adj_signal = signal_base * (prob_bull / sigma_t)

    # Position Sizing: k * (Signal_adj / Sigma)
    pos_size = risk_k * (adj_signal / sigma_t)
    pos_size = np.clip(pos_size, -1.0, 1.0) 

    # --- TOP METRICS ---
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Unified Signal", f"{adj_signal:.4f}")
    m2.metric("Regime Prob (Bull)", f"{prob_bull:.2%}")
    m3.metric("GARCH Vol (Daily)", f"{sigma_t:.4%}")
    m4.metric("Recommended Position", f"{pos_size:.2%}")

    # --- CHARTS ---
    col_left, col_right = st.columns([2, 1])

    with col_left:
        st.subheader("Price & Kalman Trend")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df.index, y=df['Adj Close'], name="Market Price", line=dict(color='gray', width=1)))
        fig.add_trace(go.Scatter(x=df.index, y=df['Adj Close'].rolling(window=len(df), min_periods=1).mean().index.map(lambda x: k_trend if x == df.index[-1] else None), name="Kalman Target", mode='markers', marker=dict(size=10, color='cyan')))
        # Visualizing the Kalman smoothed line
        kf_full = KalmanFilter(transition_matrices=[1], observation_matrices=[1]).filter(df['Adj Close'].values)[0]
        fig.add_trace(go.Scatter(x=df.index, y=kf_full.flatten(), name="Kalman Filter", line=dict(color='cyan', dash='dot')))
        st.plotly_chart(fig, use_container_width=True)

    with col_right:
        st.subheader("5-Day Monte Carlo")
        var5, conf95, paths = run_monte_carlo(df['Adj Close'].iloc[-1], sigma_t)
        mc_fig = go.Figure()
        for i in range(30): # Show subset of paths
            mc_fig.add_trace(go.Scatter(y=paths[i], mode='lines', line=dict(width=0.5), showlegend=False))
        st.plotly_chart(mc_fig, use_container_width=True)
        st.write(f"**VaR (5%):** ${var5:.2f}")
        st.write(f"**95% Target:** ${conf95:.2f}")

    # --- MODEL DATA TABLE ---
    st.subheader("Model Decision Layer")
    results_df = pd.DataFrame({
        "Feature": ["Exp Return (LR)", "Exp Return (ARIMA)", "GARCH Vol", "Regime Prob", "Kalman Trend"],
        "Value": [f"{r_lr:.6f}", f"{r_arima:.6f}", f"{sigma_t:.6f}", f"{prob_bull:.2%}", f"{k_trend:.2f}"]
    })
    st.table(results_df)

else:
    st.error("Invalid Ticker or No Data Found.")

2026-04-07 17:45:09.759 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 17:45:09.760 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 17:45:09.760 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 17:45:09.760 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 17:45:09.761 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 17:45:09.761 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 17:45:09.762 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 17:45:09.764 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar